In [72]:
import pandas as pd

ocean_data = pd.read_csv("ocean.csv")
data=pd.read_csv('labelled_shark_data.csv')
# Convert the "time" column to datetime format
ocean_data['time'] = pd.to_datetime(ocean_data['time'])

# Drop irrelevant columns
ocean_data.drop(['QC_Flag'], axis=1, inplace=True)

# Fill missing numeric values with mean
ocean_data.fillna(ocean_data.select_dtypes(include='number').mean(), inplace=True)
# Convert the "datetime" column in the shark dataset to datetime64[ns, UTC] type
data['datetime'] = pd.to_datetime(data['datetime'], format='%d/%m/%Y %H:%M')
ocean_data['time'] = ocean_data['time'].dt.tz_localize(None)

# Extract date from datetime in the shark dataset
data['date'] = data['datetime'].dt.date

# Extract date from time in the ocean dataset
ocean_data['date'] = ocean_data['time'].dt.date


# Group ocean data by date and compute aggregate statistics
aggregate_columns = ['AtmosphericPressure', 'WindDirection', 'WindSpeed', 'Gust',
                     'WaveHeight', 'WavePeriod','AirTemperature', 'DewPoint', 'SeaTemperature', 'RelativeHumidity']

# Group ocean data by date and compute aggregate statistics
ocean_aggregated = ocean_data.groupby('date')[aggregate_columns].mean().reset_index()


# Merge aggregated ocean data with shark data based on date
merged_data = pd.merge(data, ocean_aggregated, on='date', how='left')

# View the merged data
print(merged_data)

merged_data.drop(columns=['date'], inplace=True)
data=merged_data.copy(0)
data['datetime'] = pd.to_datetime(data['datetime'], format="%d/%m/%Y %H:%M")
data['datetime'] = data['datetime'].astype(int) // 10**9

from sklearn.discriminant_analysis import StandardScaler
from sklearn.model_selection import train_test_split
X = data[['datetime', 'long_dep', 'lat_dep','AtmosphericPressure','WindDirection','WindSpeed','Gust','WaveHeight','SeaTemperature']].values
y = data['is_illegal'].values

# Normalize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

                  datetime  animal_id       Time   long_dep   lat_dep  \
0      2014-01-01 00:00:00       1061   23:58:38  71.940121 -5.517135   
1      2014-01-01 00:04:00       1061   00:04:28  71.940121 -5.517135   
2      2014-01-01 00:05:00       1061   00:04:28  71.940121 -5.517135   
3      2014-01-01 01:21:00       1061   01:21:45  71.940121 -5.517135   
4      2014-01-01 01:23:00       1061   01:21:45  71.940121 -5.517135   
...                    ...        ...        ...        ...       ...   
356785 2015-12-31 21:23:00       1325   21:23:50  72.051417 -5.662917   
356786 2015-12-31 23:41:00       1204   23:41:54  71.764045 -5.445147   
356787 2015-12-31 23:48:00       1207   23:48:37  72.227200 -5.336850   
356788 2015-12-31 23:49:00       1204   23:49:43  71.764045 -5.445147   
356789 2015-12-31 23:50:00       1207   23:50:06  72.227200 -5.336850   

        is_illegal        date  AtmosphericPressure  WindDirection  WindSpeed  \
0                0  2014-01-01           9

In [6]:
X

array([[ 1.38853440e+09,  7.19401210e+01, -5.51713470e+00, ...,
         2.74002623e+01,  2.93347464e+00,  1.08389839e+01],
       [ 1.38853464e+09,  7.19401210e+01, -5.51713470e+00, ...,
         2.74002623e+01,  2.93347464e+00,  1.08389839e+01],
       [ 1.38853470e+09,  7.19401210e+01, -5.51713470e+00, ...,
         2.74002623e+01,  2.93347464e+00,  1.08389839e+01],
       ...,
       [ 1.45160568e+09,  7.22272000e+01, -5.33685000e+00, ...,
         2.60301672e+01,  3.60494336e+00,  1.11930556e+01],
       [ 1.45160574e+09,  7.17640450e+01, -5.44514667e+00, ...,
         2.60301672e+01,  3.60494336e+00,  1.11930556e+01],
       [ 1.45160580e+09,  7.22272000e+01, -5.33685000e+00, ...,
         2.60301672e+01,  3.60494336e+00,  1.11930556e+01]])

# SIAMESE model on fish+envi+iuu

In [42]:
from keras.models import Model
from keras.layers import Input, Dense, Lambda
import keras.backend as K
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import  Dense
import tensorflow.keras as keras
import tensorflow as tf

def base_network(input_shape):
    inputs = Input(shape=input_shape)
    x = Dense(128, activation='relu')(inputs)
    x = Dense(64, activation='relu')(x)
    x = Dense(32, activation='relu')(x)
    x = Dense(16, activation='relu')(x)
    outputs = Dense(16)(x)
    return Model(inputs, outputs)

# Define the input shapes
input_shape = (9,)  # Assuming 3 features

# Create the left and right inputs
left_input = Input(shape=input_shape)
right_input = Input(shape=input_shape)

# Create the twin networks
base_network = base_network(input_shape)
encoded_left = base_network(left_input)
encoded_right = base_network(right_input)
# Assuming you're using TensorFlow.K for mathematical operations

def euclidean_distance(vectors):
  # Calculate Euclidean distance (ensure this function returns a single tensor)
  distance = tf.math.sqrt(tf.math.reduce_sum(tf.math.square(vectors[0] - vectors[1]), axis=1, keepdims=True))
  return distance


distance = Lambda(euclidean_distance, output_shape=(1,))(  # Specify output_shape
  [encoded_left, encoded_right])


# Create the Siamese model
siamese_model = Model(inputs=[left_input, right_input], outputs=distance)
from tensorflow.keras.optimizers import Adam, RMSprop, SGD  # Import additional optimizers

#siamese_model.compile(optimizer='adam', loss='mse')  # Using mean squared error for distance
siamese_model.compile(optimizer=Adam(learning_rate=0.0001), loss='mse')  # Example with RMSprop

import numpy as np

def create_pairs(X, y, num_classes):
    pairs = []
    labels = []
    class_indices = [np.where(y == i)[0] for i in range(num_classes)]
    min_class_samples = min(len(class_indices[i]) for i in range(num_classes))
    
    for c in range(num_classes):
        for i in range(min_class_samples):
            idx1, idx2 = class_indices[c][i], class_indices[c][(i + 1) % min_class_samples]
            pairs += [[X[idx1], X[idx2]]]
            labels += [1]  # Similar pair
            
            # Select a different class
            neg_class = (c + np.random.randint(1, num_classes)) % num_classes
            neg_idx = class_indices[neg_class][np.random.randint(0, min_class_samples)]
            pairs += [[X[idx1], X[neg_idx]]]
            labels += [0]  # Dissimilar pair
    
    return np.array(pairs), np.array(labels)


# Normalize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
# Assuming X_train and y_train are your training data and labels
num_classes = len(np.unique(y_train))
X_train_pairs, y_train_pairs = create_pairs(X_train, y_train, num_classes)
# Assuming X_val and y_val are your validation data and labels
X_val_pairs, y_val_pairs = create_pairs(X_test, y_test, num_classes)

siamese_model.fit([X_train_pairs[:, 0], X_train_pairs[:, 1]], y_train_pairs, 
                  batch_size=32, epochs=400, 
                  validation_data=([X_val_pairs[:, 0], X_val_pairs[:, 1]], y_val_pairs))


Epoch 1/400
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.2825 - val_loss: 0.2692
Epoch 2/400
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 1s 928us/step - loss: 0.2671 - val_loss: 0.2667
Epoch 3/400
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 1s 946us/step - loss: 0.2642 - val_loss: 0.2650
Epoch 4/400
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 1s 995us/step - loss: 0.2617 - val_loss: 0.2642
Epoch 5/400
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 1s 884us/step - loss: 0.2597 - val_loss: 0.2629
Epoch 6/400
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 1s 993us/step - loss: 0.2600 - val_loss: 0.2626
Epoch 7/400
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 1s 832us/step - loss: 0.2582 - val_loss: 0.2622
Epoch 8/400
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 1s 872us/step - loss: 0.2569 - val_loss: 0.2617
Epoch 9/400
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 1s 894us/step - loss: 0.2569 - val_loss: 0.2615
Epoch 10/400
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.2559 - val_loss: 0.2616
Epoch 11/400
1372/1372 ━━━━━━━━━━━━━━━━━━━━ 1s 948us/step - loss: 0.2557 - val_loss: 0.2612
E

In [43]:
predictions = siamese_model.predict([X_val_pairs[:, 0], X_val_pairs[:, 1]])
binary_predictions = (predictions > 0.5).astype(int)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, mean_absolute_error, mean_squared_error

# For Binary Classification
accuracy = accuracy_score(y_val_pairs, binary_predictions)
precision, recall, f1, _ = precision_recall_fscore_support(y_val_pairs, binary_predictions, average='binary')
cm = confusion_matrix(y_val_pairs, binary_predictions)

# For Regression
mae = mean_absolute_error(y_val_pairs, predictions)
mse = mean_squared_error(y_val_pairs, predictions)

# Print the metrics
print("Binary Classification Metrics:")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("Confusion Matrix:")
print(cm)

print("\nRegression Metrics:")
print("Mean Absolute Error:", mae)
print("Mean Squared Error:", mse)


352/352 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Binary Classification Metrics:
Accuracy: 0.506492351476343
Precision: 0.5058316024924109
Recall: 0.5631447883315546
F1-score: 0.5329517717363859
Confusion Matrix:
[[2529 3093]
 [2456 3166]]

Regression Metrics:
Mean Absolute Error: 0.4979736505289695
Mean Squared Error: 0.2627748524197773


In [45]:
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Define the features and target variable names
features = ['datetime', 'long_dep', 'lat_dep','AtmosphericPressure',
            'WindDirection', 'WindSpeed', 'Gust', 'WaveHeight', 'SeaTemperature']
target = 'is_illegal'

# Load your data from a pandas DataFrame (replace 'data' with your actual DataFrame)
X = data[features].values
y = data[target].values

# Split data into training and validation sets (e.g., 80% training, 20% validation)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features (optional but recommended)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
def siamese_model():
  # Define the input layer for each branch
  input_a = keras.layers.Input(shape=(X_train.shape[1],))
  input_b = keras.layers.Input(shape=(X_train.shape[1],))

  # Shared hidden layers
  shared_layer1 = keras.layers.Dense(256, activation='relu')(input_a)
  shared_layer1 = keras.layers.BatchNormalization()(shared_layer1)

  shared_layer2 = keras.layers.Dense(128, activation='relu')(shared_layer1)
  shared_layer2 = keras.layers.BatchNormalization()(shared_layer2)

  # Individual branches after shared layers
  encoded_a = shared_layer2  # Assign the output tensor directly
  encoded_b = shared_layer2  # Assign the output tensor directly

  # Distance layer to measure similarity between encoded outputs
  distance_layer = keras.layers.Lambda(lambda x: abs(x[0] - x[1]))([encoded_a, encoded_b])

  # Output layer (sigmoid for probability between 0 and 1)
  output = keras.layers.Dense(1, activation='sigmoid')(distance_layer)

  # Define the Siamese model with shared weights for efficiency
  model = keras.Model(inputs=[input_a, input_b], outputs=output)

  return model

# Define the Siamese model
model = siamese_model()

# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
model.fit([X_train[:, :], X_train[:, :]], y_train, epochs=50, batch_size=32, validation_data=([X_val[:, :], X_val[:, :]], y_val))

# Evaluate the model on the validation set
loss, accuracy = model.evaluate(X_val[:, :], y_val)
print("Validation Loss:", loss)
print("Validation Accuracy:", accuracy)


Epoch 1/50
8920/8920 ━━━━━━━━━━━━━━━━━━━━ 12s 1ms/step - accuracy: 0.9617 - loss: 0.3402 - val_accuracy: 0.9606 - val_loss: 0.1660
Epoch 2/50
8920/8920 ━━━━━━━━━━━━━━━━━━━━ 11s 1ms/step - accuracy: 0.9614 - loss: 0.1636 - val_accuracy: 0.9606 - val_loss: 0.1660
Epoch 3/50
8920/8920 ━━━━━━━━━━━━━━━━━━━━ 10s 1ms/step - accuracy: 0.9617 - loss: 0.1626 - val_accuracy: 0.9606 - val_loss: 0.1661
Epoch 4/50
8920/8920 ━━━━━━━━━━━━━━━━━━━━ 13s 1ms/step - accuracy: 0.9611 - loss: 0.1645 - val_accuracy: 0.9606 - val_loss: 0.1661
Epoch 5/50
8920/8920 ━━━━━━━━━━━━━━━━━━━━ 11s 1ms/step - accuracy: 0.9618 - loss: 0.1622 - val_accuracy: 0.9606 - val_loss: 0.1660
Epoch 6/50
8920/8920 ━━━━━━━━━━━━━━━━━━━━ 11s 1ms/step - accuracy: 0.9620 - loss: 0.1615 - val_accuracy: 0.9606 - val_loss: 0.1660
Epoch 7/50
8920/8920 ━━━━━━━━━━━━━━━━━━━━ 12s 1ms/step - accuracy: 0.9615 - loss: 0.1632 - val_accuracy: 0.9606 - val_loss: 0.1660
Epoch 8/50
8920/8920 ━━━━━━━━━━━━━━━━━━━━ 12s 1ms/step - accuracy: 0.9619 - loss: 0

ValueError: Layer 'functional_99' expected 2 input(s). Received 1 instead.

In [52]:
from tensorflow import keras

# Define features and target variable names
features = ['datetime', 'long_dep', 'lat_dep','AtmosphericPressure',
            'WindDirection', 'WindSpeed', 'Gust', 'WaveHeight', 'SeaTemperature']
target = 'is_illegal'

# Load your data from a pandas DataFrame (replace 'data' with your actual DataFrame)
X = data[features].values.reshape(-1, 1, X.shape[1])  # Reshape for sequence input
y = data[target].values

# Split data into training and validation sets
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the RNN model
model = keras.Sequential([
    keras.layers.LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),  # First LSTM layer with 64 units
    keras.layers.LSTM(32),  # Second LSTM layer with 32 units
    keras.layers.Dense(1, activation='sigmoid')  # Output layer with sigmoid for binary classification
])

# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_val, y_val))

# Evaluate the model on the validation set
loss, accuracy = model.evaluate(X_val, y_val)
print("Validation Loss:", loss)
print("Validation Accuracy:", accuracy)


ValueError: Found input variables with inconsistent numbers of samples: [3211110, 356790]

# cnn

In [56]:
import tensorflow as tf
from tensorflow import keras
X = data[['datetime', 'long_dep', 'lat_dep','AtmosphericPressure','WindDirection','WindSpeed','Gust','WaveHeight','SeaTemperature']].values
y = data['is_illegal'].values

# Define input shape based on your data
input_shape = (X.shape[1:])

# Build the CNN model
def cnn_model(input_shape):
  inputs = keras.Input(shape=input_shape)
  # Reshape for CNN (if needed based on your data shape)
  x = keras.layers.Reshape((X.shape[1], 1))(inputs)  # Assuming your features represent a sequence
  # Convolutional layers
  x = keras.layers.Conv1D(filters=32, kernel_size=3, activation="relu", padding="same")(x)
  x = keras.layers.MaxPooling1D(pool_size=2)(x)
  x = keras.layers.Conv1D(filters=64, kernel_size=3, activation="relu", padding="same")(x)
  x = keras.layers.MaxPooling1D(pool_size=2)(x)
  # Flatten layer
  x = keras.layers.Flatten()(x)
  # Dense layers
  x = keras.layers.Dense(128, activation="relu")(x)
  outputs = keras.layers.Dense(1, activation="sigmoid")(x)  # Sigmoid for binary classification
  return keras.Model(inputs=inputs, outputs=outputs)

# Create the model
model = cnn_model(input_shape)

# Compile the model
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# Train the model on your data
model.fit(X, y, epochs=10, batch_size=32)

# Make predictions on new data (optional)
new_data = X[0]  # Sample data point
prediction = model.predict(np.expand_dims(new_data, axis=0))
# Interpret results based on your thresholds for illegality probability


Epoch 1/10
11150/11150 ━━━━━━━━━━━━━━━━━━━━ 13s 1ms/step - accuracy: 0.9404 - loss: 152086.8281
Epoch 2/10
11150/11150 ━━━━━━━━━━━━━━━━━━━━ 12s 1ms/step - accuracy: 0.9611 - loss: 8.7590
Epoch 3/10
11150/11150 ━━━━━━━━━━━━━━━━━━━━ 11s 1ms/step - accuracy: 0.9616 - loss: 0.8794
Epoch 4/10
11150/11150 ━━━━━━━━━━━━━━━━━━━━ 11s 1ms/step - accuracy: 0.9612 - loss: 0.2926
Epoch 5/10
11150/11150 ━━━━━━━━━━━━━━━━━━━━ 11s 1ms/step - accuracy: 0.9615 - loss: 0.1614
Epoch 6/10
11150/11150 ━━━━━━━━━━━━━━━━━━━━ 11s 1ms/step - accuracy: 0.9613 - loss: 0.1638
Epoch 7/10
11150/11150 ━━━━━━━━━━━━━━━━━━━━ 12s 1ms/step - accuracy: 0.9613 - loss: 0.1640
Epoch 8/10
11150/11150 ━━━━━━━━━━━━━━━━━━━━ 11s 1ms/step - accuracy: 0.9617 - loss: 0.1626
Epoch 9/10
11150/11150 ━━━━━━━━━━━━━━━━━━━━ 12s 1ms/step - accuracy: 0.9615 - loss: 0.1633
Epoch 10/10
11150/11150 ━━━━━━━━━━━━━━━━━━━━ 11s 1ms/step - accuracy: 0.9615 - loss: 0.1635
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step


In [58]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator

# Define residual block (same as previous example)
def residual_block(x, filters, strides=(1, 1)):
  shortcut = x
  x = keras.layers.Conv2D(filters=filters, kernel_size=3, strides=strides, padding="same", activation="relu")(x)
  x = keras.layers.Conv2D(filters=filters, kernel_size=3, strides=(1, 1), padding="same")(x)
  x = tf.keras.layers.Add()([x, shortcut])
  x = keras.layers.Activation("relu")(x)
  return x

# Define ResNet50 model (same as previous example)
def ResNet50(input_shape=(32, 32, 3), classes=10):
  inputs = keras.Input(shape=input_shape)
  x = keras.layers.Conv2D(filters=64, kernel_size=7, strides=(2, 2), padding="same", activation="relu")(inputs)
  x = keras.layers.MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding="same")(x)
  # ... (Residual stages and global average pooling) ...
  outputs = keras.layers.Dense(classes, activation="softmax")(x)
  model = keras.Model(inputs=inputs, outputs=outputs)
  return model

# Data preparation (assuming your data is NumPy arrays)
# Separate features (X) and labels (y)
X = data[['datetime', 'long_dep', 'lat_dep','AtmosphericPressure','WindDirection','WindSpeed','Gust','WaveHeight','SeaTemperature']].values.reshape(-1, 1, X.shape[1])  # Reshape for CNN (assuming features as a sequence)
y = data['is_illegal'].values

# Define training and validation data generators (adjust parameters as needed)
batch_size = 32
sequence_length = X.shape[1]  # Length of your sequence (implicitly used during data preparation)

train_generator = TimeseriesGenerator(X, y, batch_size=batch_size, shuffle=True)
validation_generator = TimeseriesGenerator(X, y, batch_size=batch_size, shuffle=False)

# Create the ResNet50 model
model = ResNet50(input_shape=(sequence_length, X.shape[2], 1))  # Adjust input shape based on your data

# Compile the model (adjust optimizer and loss as needed)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# Train the model
model.fit(train_generator, epochs=10, validation_data=validation_generator)

# Make predictions on new data (optional)
new_data_sequence = X[-sequence_length:]  # Assuming the last sequence element is for prediction
new_data_sequence = new_data_sequence.reshape(1, sequence_length, X.shape[2])  # Reshape for prediction

prediction = model.predict(new_data_sequence)
predicted_class = np.argmax(prediction)  # Get the index of the predicted class

# Interpret the prediction based on your class labels (e.g., 0 - legal, 1 - illegal)
print(f"Predicted class: {predicted_class}")


TypeError: __init__() missing 1 required positional argument: 'length'

In [60]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from sklearn.preprocessing import MinMaxScaler  # For normalization

# Define residual block (same as previous example)
def residual_block(x, filters, strides=(1, 1)):
  shortcut = x
  x = keras.layers.Conv2D(filters=filters, kernel_size=3, strides=strides, padding="same", activation="relu")(x)
  x = keras.layers.Conv2D(filters=filters, kernel_size=3, strides=(1, 1), padding="same")(x)
  x = tf.keras.layers.Add()([x, shortcut])
  x = keras.layers.Activation("relu")(x)
  return x

# Define ResNet50 model (same as previous example)
def ResNet50(input_shape=(32, 32, 3), classes=10):
  inputs = keras.Input(shape=input_shape)
  x = keras.layers.Conv2D(filters=64, kernel_size=7, strides=(2, 2), padding="same", activation="relu")(inputs)
  x = keras.layers.MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding="same")(x)
  # ... (Residual stages and global average pooling) ...
  outputs = keras.layers.Dense(classes, activation="softmax")(x)
  model = keras.Model(inputs=inputs, outputs=outputs)
  return model

# Data preparation
data = data.copy()  # Avoid modifying original data

# Handle datetime feature (consider feature engineering)
# Option 1: Extract relevant information (e.g., hour, day of week)
# data['hour'] = pd.to_datetime(data['datetime']).dt.hour
# data['day_of_week'] = pd.to_datetime(data['datetime']).dt.dayofweek
# Option 2: One-hot encode the datetime (if many unique values)
# ... (One-hot encoding code)

# Normalize numerical features (excluding datetime features)
scaler = MinMaxScaler()
numerical_features = ['long_dep', 'lat_dep', 'AtmosphericPressure', 'WindDirection', 'WindSpeed', 'Gust', 'WaveHeight', 'SeaTemperature']
data[numerical_features] = scaler.fit_transform(data[numerical_features])

# Separate features (X) and labels (y)
X = data[['long_dep', 'lat_dep', 'AtmosphericPressure','WindDirection','WindSpeed','Gust','WaveHeight','SeaTemperature']].values.reshape(-1, 1, X.shape[1] - 1)  # Reshape for CNN (assuming features as a sequence)
y = data['is_illegal'].values

# Define training and validation data generators (adjust parameters as needed)
batch_size = 32
sequence_length = X.shape[1]  # Length of your sequence (consider past time steps)

train_generator = TimeseriesGenerator(X, y, batch_size=batch_size, length=sequence_length, shuffle=True)
validation_generator = TimeseriesGenerator(X, y, batch_size=batch_size, length=sequence_length, shuffle=False)

# Create the ResNet50 model
model = ResNet50(input_shape=(sequence_length, X.shape[2], 1))  # Adjust input shape based on your data

# Compile the model (adjust optimizer and loss as needed)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# Train the model
model.fit(train_generator, epochs=10, validation_data=validation_generator)

# Make predictions on new data (optional)
new_data_sequence = X[-sequence_length:]  # Assuming the last sequence element is for prediction
new_data_sequence = new_data_sequence.reshape(1, sequence_length, X.shape[2])  # Reshape for prediction

prediction = model.predict(new_data_sequence)
predicted_class = np.argmax(prediction)  # Get the index of the predicted class

# Interpret the prediction based on your class labels (e.g., 0 - legal, 1 - illegal)
print(f"Predicted class: {predicted_class}")


ValueError: cannot reshape array of size 2854320 into shape (1,0)

In [68]:
data

,datetime,animal_id,Time,long_dep,lat_dep,is_illegal,AtmosphericPressure,WindDirection,WindSpeed,Gust,WaveHeight,WavePeriod,AirTemperature,DewPoint,SeaTemperature,RelativeHumidity,hour,day_of_week
0,1388534400,1061,23:58:38,0.935014,0.886514,0,0.237538,0.636405,0.609298,0.597842,0.360609,6.632989,9.334552,7.696120,0.287781,81.94101,0,3
1,1388534640,1061,00:04:28,0.935014,0.886514,0,0.237538,0.636405,0.609298,0.597842,0.360609,6.632989,9.334552,7.696120,0.287781,81.94101,0,3
2,1388534700,1061,00:04:28,0.935014,0.886514,0,0.237538,0.636405,0.609298,0.597842,0.360609,6.632989,9.334552,7.696120,0.287781,81.94101,0,3
3,1388539260,1061,01:21:45,0.935014,0.886514,0,0.237538,0.636405,0.609298,0.597842,0.360609,6.632989,9.334552,7.696120,0.287781,81.94101,0,3
4,1388539380,1061,01:21:45,0.935014,0.886514,0,0.237538,0.636405,0.609298,0.597842,0.360609,6.632989,9.334552,7.696120,0.287781,81.94101,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356785,1451596980,1325,21:23:50,0.943188,0.825368,0,0.485182,0.630929,0.604058,0.561920,0.449891,7.046827,8.789449,8.600431,0.324075,80.46928,0,3
356786,1451605260,1204,23:41:54,0.922083,0.916708,0,0.485182,0.630929,0.604058,0.561920,0.449891,7.046827,8.789449,8.600431,0.324075,80.46928,0,3
356787,1451605680,1207,23:48:37,0.956098,0.962131,0,0.485182,0.630929,0.604058,0.561920,0.449891,7.046827,8.789449,8.600431,0.324075,80.46928,0,3
356788,1451605740,1204,23:49:43,0.922083,0.916708,0,0.485182,0.630929,0.604058,0.561920,0.449891,7.046827,8.789449,8.600431,0.324075,80.46928,0,3


In [75]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from sklearn.preprocessing import MinMaxScaler  # For normalization
from datetime import datetime

def residual_block(x, filters, strides=(1, 1)):
    shortcut = x
    
    # Main path
    x = keras.layers.Conv2D(filters=filters, kernel_size=3, strides=strides, padding="same", activation="relu")(x)
    x = keras.layers.Conv2D(filters=filters, kernel_size=3, strides=(1, 1), padding="same")(x)
    
    # Projection shortcut
    if strides != (1, 1) or shortcut.shape[-1] != filters:
        shortcut = keras.layers.Conv2D(filters=filters, kernel_size=1, strides=strides, padding="same")(shortcut)
    
    # Element-wise addition
    x = tf.keras.layers.Add()([x, shortcut])
    x = keras.layers.Activation("relu")(x)
    
    return x


def ResNet50(input_shape=(32, 32, 3), classes=10):
    inputs = keras.Input(shape=input_shape)
    x = keras.layers.Conv2D(filters=64, kernel_size=7, strides=(2, 2), padding="same", activation="relu")(inputs)
    x = keras.layers.MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding="same")(x)
    
    # First residual stage
    x = residual_block(x, filters=64, strides=(1, 1))
    x = residual_block(x, filters=64, strides=(1, 1))
    x = residual_block(x, filters=64, strides=(1, 1))

    # Second residual stage
    x = residual_block(x, filters=128, strides=(2, 2))
    x = residual_block(x, filters=128, strides=(1, 1))
    x = residual_block(x, filters=128, strides=(1, 1))
    x = residual_block(x, filters=128, strides=(1, 1))

    # Third residual stage
    x = residual_block(x, filters=256, strides=(2, 2))
    x = residual_block(x, filters=256, strides=(1, 1))
    x = residual_block(x, filters=256, strides=(1, 1))
    x = residual_block(x, filters=256, strides=(1, 1))
    x = residual_block(x, filters=256, strides=(1, 1))
    x = residual_block(x, filters=256, strides=(1, 1))

    # Fourth residual stage
    x = residual_block(x, filters=512, strides=(2, 2))
    x = residual_block(x, filters=512, strides=(1, 1))
    x = residual_block(x, filters=512, strides=(1, 1))

    # Global average pooling
    x = keras.layers.GlobalAveragePooling2D()(x)
    
    # Output layer
    outputs = keras.layers.Dense(classes, activation="softmax")(x)
    
    # Create model
    model = keras.Model(inputs=inputs, outputs=outputs)
    
    return model

# Data preparation
data = data.copy()  # Avoid modifying original data

# Handle datetime feature (feature engineering)
# Option 1: Extract hour and day of week
data['hour'] = pd.to_datetime(data['datetime']).dt.hour
data['day_of_week'] = pd.to_datetime(data['datetime']).dt.dayofweek  # 0 (Monday) - 6 (Sunday)

# Normalize numerical features (excluding datetime features)
scaler = MinMaxScaler()
numerical_features = ['long_dep', 'lat_dep', 'AtmosphericPressure', 'WindDirection', 'WindSpeed', 'Gust', 'WaveHeight', 'SeaTemperature']
data[numerical_features] = scaler.fit_transform(data[numerical_features])

# Separate features for sequences and labels
X = data[numerical_features].values  # Assuming all numerical features are used in sequences
y = data['is_illegal'].values

# Create sequences (assuming you want sequences of multiple time steps)
sequence_length = 10  # Adjust this based on your data and needs
X_sequences = []
for i in range(len(X) - sequence_length + 1):
  sequence = X[i:i + sequence_length]
  X_sequences.append(sequence)

# Convert sequences to NumPy array
X_sequences = np.array(X_sequences)

# Reshape X_sequences to match the expected input shape of the model
X_sequences = X_sequences.reshape(-1, sequence_length, X.shape[1], 1)

# Trim y to match the length of X_sequences
y_trimmed = y[:len(X_sequences)]


# Define training and validation data generators (adjust parameters as needed)
batch_size = 32
sequence_length = X_sequences.shape[1]  # Get sequence length from the data
# Define training and validation data generators with the trimmed y
train_generator = TimeseriesGenerator(X_sequences[:int(len(X_sequences) * 0.8)], y_trimmed[:int(len(y_trimmed) * 0.8)], batch_size=batch_size, length=sequence_length, shuffle=True)
validation_generator = TimeseriesGenerator(X_sequences[int(len(X_sequences) * 0.8):], y_trimmed[int(len(y_trimmed) * 0.8):], batch_size=batch_size, length=sequence_length, shuffle=False)


# Create the ResNet50 model
model = ResNet50(input_shape=(sequence_length, X.shape[1], 1))  # Adjust input shape based on your data

# Compile the model (adjust optimizer and loss as needed)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# Train the model
model.fit(train_generator, epochs=10, validation_data=validation_generator)
# Make predictions on new data (optional)
new_data_sequence = X_sequences[-1:]  # Assuming the last sequence element is for prediction
new_data_sequence = new_data_sequence.reshape(1, sequence_length, X.shape[1], 1)  # Reshape for prediction

prediction = model.predict(new_data_sequence)
predicted_class = np.argmax(prediction)  # Get the index of the predicted class

# Interpret the prediction based on your class labels (e.g., 0 - legal, 1 - illegal)
print(f"Predicted class: {predicted_class}")



Epoch 1/10


ValueError: Input 0 of layer "functional_114" is incompatible with the layer: expected shape=(None, 10, 8, 1), found shape=(None, 10, 10, 8)

In [67]:
y.shape

(356790,)

In [78]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
X = data[['datetime', 'long_dep', 'lat_dep','AtmosphericPressure','WindDirection','WindSpeed','Gust','WaveHeight','SeaTemperature']].values
y = data['is_illegal'].values


# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the Random Forest model
model = RandomForestClassifier(n_estimators=50, random_state=42)  # Adjust n_estimators as needed

# Train the model
model.fit(X_train, y_train)

# Make predictions on the testing set
y_pred = model.predict(X_test)

# Evaluate model performance (accuracy, precision, recall, etc.)
from sklearn.metrics import accuracy_score, precision_score, recall_score

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")



Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
